# Semantic Layer With OntologyRAG vs. Without — Evaluation Comparison

Does giving the ontology-generation agent access to **OntologyRAG** (a Bedrock Knowledge Base
of ontology design patterns, retrieved via `retrieve_ontology_patterns`) produce a better VKG
semantic layer than building **without** those patterns?

This notebook compares the VKG → ontology-query eval pipeline across two arms over the same
normalized S3 source data, on accuracy / latency / token usage:

1. **Run A — with OntologyRAG**: the OntologyPatterns KB is populated. This is the *normal/live*
   build condition, so by **default this arm REUSES an existing completed VKG layer** (built by
   notebook 5, or a prior run) rather than rebuilding. Then it evals the ontology query agent
   over the groundtruth dataset.
2. **Run B — without OntologyRAG**: temporarily empty the OntologyPatterns KB data source (so
   `retrieve_ontology_patterns` returns nothing) and **build the one new VKG layer** — over the
   *same source tables* as Run A's layer — then eval again.

Finally it restores the patterns and compares the two outcomes.

> **Layer reuse (default):** only the WITHOUT-OntologyRAG layer is genuinely new (it requires
> the patterns KB wiped, a state no existing layer was built in). The WITH-OntologyRAG layer is
> an ordinary VKG layer, so Run A reuses the most recent completed one by default — a run then
> builds **exactly one** layer. Set `REUSE_EXISTING_LAYER=0` to build a fresh WITH layer too, or
> pin a specific one with `VKG_LAYER_ID`. Run B always builds over Run A's table set so the only
> variable between arms is OntologyRAG on/off.

---

> ## ⚠️ DESTRUCTIVE STEP — READ BEFORE RUNNING
> Run B requires temporarily **emptying the OntologyPatterns KB S3 data source**
> (`s3://{ARTIFACTS_BUCKET}/ontology-patterns/`). This notebook does so **safely**:
> 1. it first **copies every pattern object to a timestamped backup prefix**,
> 2. deletes the live objects and re-ingests the (now empty) KB,
> 3. runs Run B,
> 4. **restores the objects from the backup** and re-ingests.
>
> The destructive cells are gated behind `CONFIRM_DESTRUCTIVE_KB_WIPE = True`. They are
> **no-ops until you set that flag**. Do not enable it against a shared/production KB unless
> you are certain, and confirm the backup step succeeded before the delete runs.

In [ ]:
# Prereq (uncomment if not already installed):
# !pip install bedrock-agentcore-starter-toolkit==0.3.9

# ── DESTRUCTIVE-OPERATION GATE ──────────────────────────────────────────────
# Run B empties the OntologyPatterns KB data source (with backup + restore).
# This is OFF by default. Set to True ONLY when you intend to run the without-
# OntologyRAG arm and understand the backup/restore flow below.
# (A full both-arms run was completed 2026-06-14; the KB was safely restored to
# its original 471 objects. Left OFF here so the notebook is safe-by-default.)
import os as _os
# Safe-by-default: stays False unless CONFIRM_DESTRUCTIVE_KB_WIPE=true is set in the
# environment for an intentional both-arms run (the wipe is backed up + restored below).
CONFIRM_DESTRUCTIVE_KB_WIPE = _os.environ.get('CONFIRM_DESTRUCTIVE_KB_WIPE', 'false').strip().lower() in ('1', 'true', 'yes')

## 1. Setup & Credentials

In [2]:
import os
import json
import uuid
import time
import boto3
import pandas as pd
from botocore.config import Config
from datetime import datetime, timezone
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv(dotenv_path='.env', override=False)
os.environ.setdefault('AWS_PROFILE', 'huthmac+demo')
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')
PROJECT_NAME = os.environ.get('PROJECT_NAME', 'semantic-layer')

invoke_config = Config(read_timeout=900, connect_timeout=60,
                       retries={'max_attempts': 0, 'mode': 'standard'}, signature_version='v4')
config = Config(retries={'max_attempts': 10, 'mode': 'standard'}, signature_version='v4')

session = boto3.Session(profile_name=os.environ['AWS_PROFILE'])
region = session.region_name or 'us-east-1'

sts = session.client('sts', region_name=region, config=config)
identity = sts.get_caller_identity()
account_id = identity['Account']
print(f"✓ AWS Credentials Verified — account ...{account_id[-4:]}, region {region}, "
      f"role {identity['Arn'].split('/')[-1]}")


import re as _re

def _redact_account_ids(obj):
    """Recursively replace AWS account IDs embedded in ARN strings with XXXXXXXXXXXX."""
    if isinstance(obj, dict):
        return {k: _redact_account_ids(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_redact_account_ids(v) for v in obj]
    if isinstance(obj, str):
        return _re.sub(r'(arn:[^:]*:[^:]*:[^:]*:)\d{12}:', r'\1XXXXXXXXXXXX:', obj)
    return obj

# Render full cell text in displayed tables — never truncate question/message/SQL
# columns (the default max_colwidth=50 cut ground-truth questions mid-word).
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)


✓ AWS Credentials Verified — account ...4087, region us-east-1, role huthmac-Isengard


In [3]:
# ── OAuth M2M runtime invoker ───────────────────────────────────────────────
# AgentCore runtimes use CUSTOM_JWT (Cognito M2M) inbound auth.
# The boto3 bedrock-agentcore client (SigV4) no longer works; use a Bearer
# token minted via Cognito client_credentials instead.
import base64
import urllib.parse as _urlparse
import urllib.request as _urlrequest

_BROWSER_UA = (
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
    'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'
)
_OAUTH_SCOPE = 'semantic-layer-mcp/invoke'


def _get_m2m_creds() -> tuple:
    """Return (token_endpoint, client_id, client_secret) from CFN + Secrets Manager.

    Returns:
        Tuple of (token_endpoint str, client_id str, client_secret str).
    """
    auth_out = stack_outputs('auth')
    token_endpoint = auth_out['McpHostedUiDomainUrl'] + '/oauth2/token'
    client_id = auth_out['M2mClientId']
    lam = session.client('lambda')
    cfg = lam.get_function_configuration(FunctionName=f'{PROJECT_NAME}-mcp-tools')
    secret_arn = cfg['Environment']['Variables']['M2M_CLIENT_SECRET_ARN']
    client_secret = session.client('secretsmanager').get_secret_value(
        SecretId=secret_arn,
    )['SecretString']
    return token_endpoint, client_id, client_secret


def _fetch_m2m_token() -> str:
    """Mint a Cognito client_credentials token for agent runtime invocations.

    Returns:
        OAuth access_token string.
    """
    token_endpoint, client_id, client_secret = _get_m2m_creds()
    body = _urlparse.urlencode({
        'grant_type': 'client_credentials',
        'scope': _OAUTH_SCOPE,
    }).encode()
    basic = base64.b64encode(f'{client_id}:{client_secret}'.encode()).decode('ascii')
    req = _urlrequest.Request(
        token_endpoint, data=body, method='POST',
        headers={
            'Content-Type': 'application/x-www-form-urlencoded',
            'Authorization': f'Basic {basic}',
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read())['access_token']


def _invoke_runtime(arn: str, session_id: str, payload: bytes, *, qualifier: str = 'DEFAULT') -> bytes:
    """Invoke an AgentCore Runtime with Cognito Bearer auth.

    Parameters:
        arn: runtime ARN.
        session_id: X-Amzn-Bedrock-AgentCore-Runtime-Session-Id header value.
        payload: JSON-encoded request body.
        qualifier: runtime qualifier (default 'DEFAULT').
    Returns:
        Raw response body bytes.
    """
    token = _fetch_m2m_token()
    encoded_arn = arn.replace(':', '%3A').replace('/', '%2F')
    url = (
        f'https://bedrock-agentcore.{region}.amazonaws.com/'
        f'runtimes/{encoded_arn}/invocations?qualifier={qualifier}'
    )
    req = _urlrequest.Request(
        url, data=payload, method='POST',
        headers={
            'Authorization': f'Bearer {token}',
            'Content-Type': 'application/json',
            'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id,
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=900) as resp:
        return resp.read()


print('✓ OAuth M2M runtime invoker ready')

✓ OAuth M2M runtime invoker ready


## 2. Resolve Stacks, Runtimes, KBs & Source Coordinates

We need the VKG build/query runtimes plus BOTH knowledge bases:
- `OntologyPatternsKbId` / `OntologyPatternsDataSourceId` — the OntologyRAG patterns KB we
  toggle on/off.
- the S3 location of the patterns (to back up / wipe / restore).

In [4]:
def stack_outputs(stack_suffix: str) -> dict:
    """Return {OutputKey: OutputValue} for '{PROJECT_NAME}-{suffix}', or {} if absent."""
    cfn = session.client('cloudformation', region_name=region)
    for name in (f'{PROJECT_NAME}-dev-{stack_suffix}', f'{PROJECT_NAME}-{stack_suffix}'):
        try:
            return {o['OutputKey']: o['OutputValue']
                    for o in cfn.describe_stacks(StackName=name)['Stacks'][0].get('Outputs', [])}
        except Exception:
            continue
    return {}

ac = stack_outputs('agentcore')
kb = stack_outputs('bedrock-kb')
datalake = stack_outputs('data-lake')

ONTOLOGY_RUNTIME_ARN = ac['OntologyRuntimeArn']
ONTOLOGY_QUERY_RUNTIME_ARN = ac['QueryRuntimeArn']

ONTOLOGY_PATTERNS_KB_ID = kb['OntologyPatternsKbId']
ONTOLOGY_PATTERNS_DS_ID = kb['OntologyPatternsDataSourceId']

S3T_DATABASE = os.environ.get('S3T_DATABASE') or datalake.get('Namespace', 'normalized')
S3T_CATALOG = os.environ.get('S3T_CATALOG')
if not S3T_CATALOG:
    prefix = datalake.get('S3TablesFederatedCatalogName', 's3tablescatalog')
    S3T_CATALOG = f"{prefix}/{PROJECT_NAME}-dev-analytics-tables"

METADATA_TABLE = os.environ.get('ONTOLOGY_METADATA_TABLE') or stack_outputs('dynamodb').get('MetadataTableName', f'{PROJECT_NAME}-dev-metadata')

# Resolve the patterns S3 location straight from the data source config.
bedrock_agent = session.client('bedrock-agent', region_name=region)
_ds = bedrock_agent.get_data_source(
    knowledgeBaseId=ONTOLOGY_PATTERNS_KB_ID, dataSourceId=ONTOLOGY_PATTERNS_DS_ID)['dataSource']
_s3c = _ds.get('dataSourceConfiguration', {}).get('s3Configuration', {})
PATTERNS_BUCKET = _s3c.get('bucketArn', '').split(':::')[-1]
PATTERNS_PREFIXES = _s3c.get('inclusionPrefixes', ['ontology-patterns/'])
PATTERNS_PREFIX = PATTERNS_PREFIXES[0] if PATTERNS_PREFIXES else 'ontology-patterns/'

print("✓ Resolved coordinates:")
print(f"  VKG build/query     = {ONTOLOGY_RUNTIME_ARN.rsplit('/',1)[-1]} / {ONTOLOGY_QUERY_RUNTIME_ARN.rsplit('/',1)[-1]}")
print(f"  OntologyPatterns KB = {ONTOLOGY_PATTERNS_KB_ID} / ds {ONTOLOGY_PATTERNS_DS_ID}")
print(f"  Patterns S3         = s3://{PATTERNS_BUCKET}/{PATTERNS_PREFIX}")
print(f"  Normalized S3       = catalog '{S3T_CATALOG}', namespace '{S3T_DATABASE}'")
print(f"  Metadata table      = {METADATA_TABLE}")

✓ Resolved coordinates:
  VKG build/query     = semantic_layer_dev_ontology-iddx7A6C74 / semantic_layer_dev_ontology_query-HKGVpkE6XK
  OntologyPatterns KB = 5HWK8RDUUR / ds UQDWVFGG5N
  Patterns S3         = s3://semantic-layer-dev-artifacts-XXXXXXXXXXXX/ontology-patterns/
  Normalized S3       = catalog 's3tablescatalog/semantic-layer-dev-analytics-tables', namespace 'normalized'
  Metadata table      = semantic-layer-dev-metadata


In [5]:
# ── OAuth M2M runtime invoker ───────────────────────────────────────────────
# AgentCore runtimes use CUSTOM_JWT (Cognito M2M) inbound auth.
# The boto3 bedrock-agentcore client (SigV4) no longer works; use a Bearer
# token minted via Cognito client_credentials instead.
import base64
import urllib.parse as _urlparse
import urllib.request as _urlrequest

_BROWSER_UA = (
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
    'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'
)
_OAUTH_SCOPE = 'semantic-layer-mcp/invoke'


def _get_m2m_creds() -> tuple:
    """Return (token_endpoint, client_id, client_secret) from CFN + Secrets Manager.

    Returns:
        Tuple of (token_endpoint str, client_id str, client_secret str).
    """
    auth_out = stack_outputs('auth')
    token_endpoint = auth_out['McpHostedUiDomainUrl'] + '/oauth2/token'
    client_id = auth_out['M2mClientId']
    lam = session.client('lambda')
    cfg = lam.get_function_configuration(FunctionName=f'{PROJECT_NAME}-mcp-tools')
    secret_arn = cfg['Environment']['Variables']['M2M_CLIENT_SECRET_ARN']
    client_secret = session.client('secretsmanager').get_secret_value(
        SecretId=secret_arn,
    )['SecretString']
    return token_endpoint, client_id, client_secret


def _fetch_m2m_token() -> str:
    """Mint a Cognito client_credentials token for agent runtime invocations.

    Returns:
        OAuth access_token string.
    """
    token_endpoint, client_id, client_secret = _get_m2m_creds()
    body = _urlparse.urlencode({
        'grant_type': 'client_credentials',
        'scope': _OAUTH_SCOPE,
    }).encode()
    basic = base64.b64encode(f'{client_id}:{client_secret}'.encode()).decode('ascii')
    req = _urlrequest.Request(
        token_endpoint, data=body, method='POST',
        headers={
            'Content-Type': 'application/x-www-form-urlencoded',
            'Authorization': f'Basic {basic}',
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read())['access_token']


def _invoke_runtime(arn: str, session_id: str, payload: bytes, *, qualifier: str = 'DEFAULT') -> bytes:
    """Invoke an AgentCore Runtime with Cognito Bearer auth.

    Parameters:
        arn: runtime ARN.
        session_id: X-Amzn-Bedrock-AgentCore-Runtime-Session-Id header value.
        payload: JSON-encoded request body.
        qualifier: runtime qualifier (default 'DEFAULT').
    Returns:
        Raw response body bytes.
    """
    token = _fetch_m2m_token()
    encoded_arn = arn.replace(':', '%3A').replace('/', '%2F')
    url = (
        f'https://bedrock-agentcore.{region}.amazonaws.com/'
        f'runtimes/{encoded_arn}/invocations?qualifier={qualifier}'
    )
    req = _urlrequest.Request(
        url, data=payload, method='POST',
        headers={
            'Authorization': f'Bearer {token}',
            'Content-Type': 'application/json',
            'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id,
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=900) as resp:
        return resp.read()


print('✓ OAuth M2M runtime invoker ready')

✓ OAuth M2M runtime invoker ready


## 3. Load Groundtruth & Discover Source Tables

In [6]:
with open('../data/eval/groundtruth_dataset.json', 'r') as f:
    groundtruth_all = json.load(f)
# Advisory rows (Expected_Tier == 'advisory') carry no SQL — exempt them from the
# SQL-column requirement (mirrors nb2/nb6/nb9/nb10). The VKG agent answers advisory
# questions too, so advisory rows are KEPT in this comparison.
BASE_COLS = {'Natural_Language_Question', 'Expected_Answer'}
SQL_COLS = {'Expected_SQL_Query', 'Expected_SQL_Result'}
for i, row in enumerate(groundtruth_all):
    required = BASE_COLS if row.get('Expected_Tier') == 'advisory' else (BASE_COLS | SQL_COLS)
    if required - set(row.keys()):
        raise ValueError(f"Row {i} missing columns: {required - set(row.keys())}")
MAX_ROWS = int(os.environ.get('MAX_ROWS', '2'))
groundtruth = groundtruth_all[:MAX_ROWS] if MAX_ROWS > 0 else list(groundtruth_all)

glue = session.client('glue', region_name=region)
MAX_TABLES = int(os.environ.get('MAX_TABLES', '3'))
s3_catalog_id = f"{account_id}:{S3T_CATALOG}"
s3_tables = []
for page in glue.get_paginator('get_tables').paginate(CatalogId=s3_catalog_id, DatabaseName=S3T_DATABASE):
    s3_tables.extend(t['Name'] for t in page.get('TableList', []))
s3_tables = sorted(s3_tables)
if MAX_TABLES > 0:
    s3_tables = s3_tables[:MAX_TABLES]

def s3_source(table_name: str) -> dict:
    """Data-source dict for a normalized S3 Tables (Iceberg) table."""
    return {
        'databaseName': S3T_DATABASE, 'tableName': table_name, 'catalogId': S3T_CATALOG,
        'dataSource': 'AwsDataCatalog', 'tableId': f'{S3T_CATALOG}::{S3T_DATABASE}.{table_name}',
    }

SOURCES = [s3_source(t) for t in s3_tables]
print(f"✓ {len(groundtruth)} groundtruth row(s); {len(SOURCES)} source table(s): {s3_tables}")

✓ 16 groundtruth row(s); 40 source table(s): ['address', 'admin_codes', 'annuity_detail', 'arrangement_destination', 'arrangement_source', 'carrier_appointment', 'coverage', 'coverage_product', 'distribution_level', 'email_address', 'financial_activity', 'financial_statement', 'govt_id_info', 'holding', 'holding_activity', 'holding_arrangement', 'holding_dbg', 'holding_loan', 'holding_payout', 'holding_projection', 'holding_restriction', 'holding_subaccount', 'invest_product', 'life_detail', 'life_participant', 'loan_activity', 'party', 'party_banking', 'party_license', 'phone', 'policy_loan_summary', 'policy_product', 'producer_agreement', 'reinsurance_info', 'relation', 'rider', 'rider_participant', 'subaccount_activity', 'substandard_rating', 'type_codes']


## 4. Reusable Helpers

Build/eval helpers (same as notebook 7's VKG path) plus the **OntologyPatterns KB toggle**
helpers used to switch OntologyRAG off (with backup) and back on (restore).

In [7]:
ddb = session.resource('dynamodb').Table(METADATA_TABLE)
agentcore = session.client('bedrock-agentcore', region_name=region, config=invoke_config)
s3 = session.client('s3', region_name=region)


def build_vkg_layer(*, label: str, sources: list, max_wait_per_table_s: int = 150) -> dict:
    """Seed a VKG config and run the ontology build agent to completion. Returns run metadata."""
    case_id = f"vkg-{label}-{uuid.uuid4().hex[:8]}"
    rsid = str(uuid.uuid4())
    now = datetime.now(timezone.utc).isoformat()
    ts = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
    ddb.put_item(Item={
        'id': case_id, 'version': 'v1', 'type': 'VKG', 'name': f'{label}-{ts}',
        'status': 'pending', 'configuration': {}, 'dataSources': sources,
        'createdAt': now, 'updatedAt': now, 'buildStartedAt': now,
        'createdBy': 'eval@semantic-layer.local',
        'dataSourcesDescription': f'{len(sources)} normalized table(s) for OntologyRAG on/off eval',
        'useCasesDescription': f'Insurance VKG A/B arm "{label}": OWL ontology over the curated insurance semantic layer (parties, policies, coverages, riders, payouts) for graph-grounded NL querying, built {"with" if "without" not in label else "without"} FIBO/ACORD OntologyRAG design patterns.',
    })
    print(f"  ✓ Seeded VKG config {case_id} ({len(sources)} table(s))")
    _invoke_runtime(ONTOLOGY_RUNTIME_ARN, rsid, json.dumps({'id': case_id}).encode('utf-8'))
    max_wait = max(300, max_wait_per_table_s * len(sources))
    start = datetime.now()
    status = 'processing'
    print(f"  Polling VKG build (max {max_wait}s)...")
    while (datetime.now() - start).total_seconds() < max_wait:
        time.sleep(30)
        item = ddb.get_item(Key={'id': case_id, 'version': 'v1'}).get('Item', {})
        status = item.get('status', 'unknown')
        el = (datetime.now() - start).total_seconds()
        print(f"    [{el:.0f}s] status={status} {item.get('phase','')}")
        if status in ('completed', 'failed'):
            break
    duration = (datetime.now() - start).total_seconds()
    print(f"  {'✓' if status=='completed' else '✗'} VKG build {status} in {duration:.0f}s")
    return {'case_id': case_id, 'status': status, 'duration_s': duration,
            'num_tables': len(sources), 'runtime_session_id': rsid}


# ════════════════════════════════════════════════════════════════════════════
# Server-side evaluation infrastructure — NO local scoring.
#
# Both arms evaluate the SAME ontology query agent (VKG path); only the underlying VKG layer
# differs (built with vs without OntologyRAG). Scoring runs entirely inside AgentCore Batch
# Evaluations (the pattern from notebook 2): the only client-side work is invoking the agent
# once per groundtruth row. The custom evaluators are created ONCE and reused across both arms.
# ════════════════════════════════════════════════════════════════════════════
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_runner import (
    BatchEvaluationRunner,
)
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_models import (
    BatchEvaluationRunConfig, BatchEvaluatorConfig, CloudWatchDataSourceConfig,
)
from bedrock_agentcore.evaluation.runner.dataset_types import (
    Dataset, PredefinedScenario, Turn,
)
from bedrock_agentcore.evaluation.runner.invoker_types import (
    AgentInvokerInput, AgentInvokerOutput,
)

ONTOLOGY_QUERY_AGENT_ID = ONTOLOGY_QUERY_RUNTIME_ARN.rsplit('/', 1)[-1]
_cp = session.client('bedrock-agentcore-control', region_name=region)
_SUFFIX = uuid.uuid4().hex[:8]
JUDGE_MODEL_ID = 'global.anthropic.claude-sonnet-4-6'
_BINARY_SCALE = {
    'numerical': [
        {'value': 0.0, 'label': 'fail', 'definition': 'Does not satisfy the criterion.'},
        {'value': 1.0, 'label': 'pass', 'definition': 'Fully satisfies the criterion.'},
    ]
}


def _create_llm_judge(*, name: str, level: str, instructions: str) -> str:
    """Create a binary LLM-as-Judge evaluator and return its evaluatorId.

    Parameters:
        name: human-readable evaluator name (a unique suffix is appended).
        level: 'TRACE' or 'SESSION' — the granularity the service scores at.
        instructions: judge prompt; may reference ground-truth + context placeholders.
    Returns:
        The created evaluator's evaluatorId.
    """
    resp = _cp.create_evaluator(
        evaluatorName=f"{name}_{_SUFFIX}",
        level=level,
        evaluatorConfig={
            'llmAsAJudge': {
                'instructions': instructions,
                'ratingScale': _BINARY_SCALE,
                'modelConfig': {
                    'bedrockEvaluatorModelConfig': {
                        'modelId': JUDGE_MODEL_ID,
                        'inferenceConfig': {'maxTokens': 1024},
                    }
                },
            }
        },
    )
    return resp['evaluatorId']


# SESSION-level (NOT TRACE): score the conversation's FINAL answer once (right granularity
# for a multi-turn agent). Clarify turns now always emit an answer span
# (shared/answer_span.emit_answer_span), so the old "TRACE errors on clarify → session FAILED"
# no longer applies. Expected answer threaded via a SESSION assertion (FINAL_ANSWER_ASSERTION_PREFIX).
ANSWER_FAITHFUL_ID = _create_llm_judge(
    name='FinalAnswerFaithfulness',
    level='SESSION',
    instructions=(
        "You are a strict binary evaluator for a virtual-knowledge-graph (VKG) text-to-SQL agent.\n\n"
        "Session context (full conversation across ALL turns):\n{context}\n\n"
        "Assertions about this session:\n{assertions}\n\n"
        "Exactly one assertion begins with 'The conversation's final answer matches:' — the "
        "text after that prefix is the EXPECTED final answer. Judge ONLY the FINAL substantive "
        "answer at the end of the conversation.\n\n"
        "Score 1 (pass) iff that final answer is factually consistent with the expected "
        "answer — same key numbers, entities, and conclusion. Score 0 (fail) if it "
        "contradicts it, invents figures, omits the requested result, or the conversation "
        "never reaches a substantive answer. If no assertion carries the expected-answer "
        "prefix, score 0 and note the ground truth is missing."
    ),
)

SQL_GROUNDED_ID = _create_llm_judge(
    name='SqlGrounded',
    level='SESSION',
    instructions=(
        "You are a strict binary grounding evaluator for a VKG text-to-SQL data agent.\n\n"
        "Session context (full conversation, including tool calls and tool results):\n"
        "{context}\n\n"
        "Available tools: {available_tools}\n\n"
        "This VKG agent runs a deterministic graph: it fetches the ontology, assembles an "
        "ontology SLICE, generates SPARQL, then Phase 5 translates it to SQL (Ontop) and runs "
        "it on Athena. Fetch/slice are graph phases, not model tool calls. In the context, "
        "locate (a) the RETRIEVED ONTOLOGY CONTEXT — the ontology slice (classes/properties "
        "with mapsToTable/mapsToColumn) the agent may use; and (b) the executed SQL (the "
        "Ontop-translated SQL run on Athena).\n\n"
        "Score 1 (pass) iff EVERY table, column, and join in the executed SQL maps to a "
        "class/property/mapping present in the retrieved ontology context (case-insensitive; "
        "tolerate aliases, quoted vs unquoted identifiers, and SQL builtins like COUNT/SUM/"
        "DATE_TRUNC — not schema). Score 0 (fail) if the SQL references any table/column absent "
        "from the ontology context (hallucinated schema), or if no ontology context is present. "
        "Name the first offending identifier when you fail it."
    ),
)

TOOL_ORDERING_ID = _create_llm_judge(
    name='ToolCallOrdering',
    level='SESSION',
    instructions=(
        "You are a strict binary evaluator checking whether a VKG text-to-SQL agent grounded "
        "its query in the retrieved ontology BEFORE executing SQL.\n\n"
        "This agent runs a deterministic graph, not a tool loop. Prescribed flow: 1. fetch the "
        "ontology (graph phase, appears as the ontology slice, not a tool call)  2. assemble an "
        "ontology SLICE  3. generate SPARQL against the slice, then Phase 5 translates it to "
        "SQL (Ontop) and runs it on Athena.\n\n"
        "Available tools: {available_tools}\n"
        "Session context (phase outputs + the SQL execution, chronological): {context}\n\n"
        "Score 1 (pass) iff a retrieved ontology context (the ontology slice) is present and "
        "precedes the SQL execution (skipping a fresh fetch when the ontology is supplied "
        "inline is acceptable). Score 0 (fail) if SQL is executed with no retrieved ontology "
        "context before it. Name the offending ordering when you fail it."
    ),
)

CUSTOM_EVALUATOR_IDS = [ANSWER_FAITHFUL_ID, SQL_GROUNDED_ID, TOOL_ORDERING_ID]
ALL_EVALUATOR_IDS = [
    # SESSION-only (matches notebook 2/6). Builtin.Correctness (TRACE) is DROPPED — it errors
    # on clarification turns and fails the whole session; GoalSuccessRate + the 3 SESSION
    # custom judges read the full multi-turn {context}/{assertions}.
    'Builtin.GoalSuccessRate',
    *CUSTOM_EVALUATOR_IDS,
]
_EVAL_NAME = {ANSWER_FAITHFUL_ID: 'FinalAnswerFaithfulness',
              SQL_GROUNDED_ID: 'SqlGrounded', TOOL_ORDERING_ID: 'ToolCallOrdering'}
# VKG agent runs a deterministic graph (phase fns); Phase 5 translates SPARQL→SQL (Ontop) and
# runs Athena directly — no model tool calls — so the expected trajectory is empty.
EXPECTED_TRAJECTORY = []

batch_runner = BatchEvaluationRunner(region=region)
print("✓ Custom server-side evaluators created (LLM-as-Judge, no Lambda):")
print(f"  FinalAnswerFaithfulness (SESSION) : {ANSWER_FAITHFUL_ID}")
print(f"  SqlGrounded             (SESSION) : {SQL_GROUNDED_ID}")
print(f"  ToolCallOrdering        (SESSION) : {TOOL_ORDERING_ID}")


def run_server_side_eval(*, label: str, eval_id: str) -> dict:
    """Score the ontology query agent over every groundtruth row — entirely server-side.

    One PredefinedScenario per groundtruth row, invoked against the VKG layer `eval_id`;
    AgentCore Batch Evaluations runs the built-in + custom evaluators in-service.

    Parameters:
        label: human tag for this arm ('with-ontologyrag' / 'without-ontologyrag').
        eval_id: the VKG layer config id the ontology query agent should resolve.
    Returns:
        {'label', 'eval_id', 'batch_id', 'status', 'agg', 'events', 'agent_perf', 'agent_runs'}.
    """
    invoke_client = session.client('bedrock-agentcore', region_name=region, config=invoke_config)
    scenario_by_q = {row['Natural_Language_Question']: f"gt-row-{i:02d}"
                     for i, row in enumerate(groundtruth)}
    agent_runs: dict = {}

    def agent_invoker(inp: AgentInvokerInput) -> AgentInvokerOutput:
        """Invoke the ontology query agent once for one scenario; record cost/latency."""
        question = inp.payload if isinstance(inp.payload, str) else json.dumps(inp.payload)
        sid = scenario_by_q.get(question, question)
        start = time.time()
        answer, sql_query, usage, err = '', '', {}, None
        try:
            raw = _invoke_runtime(
                ONTOLOGY_QUERY_RUNTIME_ARN,
                inp.session_id,
                json.dumps({'question': question, 'id': eval_id}).encode('utf-8'),
            )
            text = raw.decode('utf-8', errors='replace')
            data = json.loads(text) if text.strip().startswith('{') else {'answer': text}
            answer = data.get('answer', '') or ''
            sql_query = data.get('sql_query', '') or ''
            usage = (data.get('metadata') or {}).get('usage') or {}
        except Exception as exc:  # noqa: BLE001 — record and continue
            err = str(exc)
            print(f"    ⚠ [{sid}] invocation error: {err}")
        wall = time.time() - start
        agent_runs[inp.session_id] = {
            'scenario_id': sid, 'question': question, 'agent_sql': sql_query,
            'total_tokens': usage.get('totalTokens'), 'wall_clock_s': round(wall, 2),
            'invoke_error': err,
        }
        print(f"    {'✓' if err is None else '✗'} [{sid}] {wall:.1f}s tokens={usage.get('totalTokens')}")
        return AgentInvokerOutput(agent_output=answer)

    dataset = Dataset(scenarios=[
        PredefinedScenario(
            scenario_id=f"gt-row-{i:02d}",
            turns=[Turn(input=row['Natural_Language_Question'],
                        expected_response=row['Expected_Answer'])],
            expected_trajectory=EXPECTED_TRAJECTORY,
            assertions=[
                f"The conversation's final answer matches: {row['Expected_Answer']}",
                "The agent grounded its query in the fetched ontology slice, then executed SQL on Athena.",
                # Advisory rows carry no SQL — .get() so they don't KeyError.
                f"The executed SQL is logically equivalent to: {row.get('Expected_SQL_Query', '')}",
                f"The result set matches: {json.dumps(row.get('Expected_SQL_Result', ''))[:500]}",
            ],
        )
        for i, row in enumerate(groundtruth)
    ])

    svc = f"{ONTOLOGY_QUERY_AGENT_ID}.DEFAULT"
    cfg = BatchEvaluationRunConfig(
        batch_evaluation_name=f"cmp_{label}_{uuid.uuid4().hex[:8]}".replace('-', '_'),
        description=f"Server-side eval of the ontology query agent over the {label} layer.",
        evaluator_config=BatchEvaluatorConfig(evaluator_ids=ALL_EVALUATOR_IDS),
        data_source=CloudWatchDataSourceConfig(
            service_names=[svc],
            log_group_names=['aws/spans',
                             f"/aws/bedrock-agentcore/runtimes/{ONTOLOGY_QUERY_AGENT_ID}-DEFAULT"],
            ingestion_delay_seconds=180,
        ),
        max_concurrent_scenarios=3, polling_timeout_seconds=3600, polling_interval_seconds=30,
    )
    print(f"  Running server-side batch eval for '{label}' ({len(dataset.scenarios)} scenarios)...")
    result = batch_runner.run_dataset_evaluation(config=cfg, dataset=dataset, agent_invoker=agent_invoker)
    print(f"  ✓ Batch status: {result.status}")

    agg = {}
    ev = result.evaluation_results
    if ev is not None:
        for es in (ev.evaluator_summaries or []):
            st = es.statistics
            name = _EVAL_NAME.get(es.evaluator_id, es.evaluator_id)
            agg[name] = (round(st.average_score, 3)
                         if st and st.average_score is not None else None)

    events = []
    for _ in range(6):
        try:
            events = batch_runner.fetch_evaluation_events(result)
            if events:
                break
        except (LookupError, ValueError):
            pass
        time.sleep(20)

    def _f(e, k):
        return e[k] if k in e else (e.get('attributes', {}) or {}).get(k)
    event_rows = []
    for e in events:
        sess = _f(e, 'session.id') or _f(e, 'gen_ai.session.id')
        run = agent_runs.get(sess, {})
        event_rows.append({
            'arm': label, 'scenario_id': run.get('scenario_id'),
            'evaluator': _EVAL_NAME.get(_f(e, 'gen_ai.evaluation.name'), _f(e, 'gen_ai.evaluation.name')),
            'score': _f(e, 'gen_ai.evaluation.score.value'),
            'label_': _f(e, 'gen_ai.evaluation.score.label'),
        })

    _lat = [r['wall_clock_s'] for r in agent_runs.values() if r.get('wall_clock_s') is not None]
    agent_perf = {
        'rows': len(agent_runs),
        'avg_wall_clock_s': round(sum(_lat) / len(_lat), 2) if _lat else None,
        'agent_total_tokens': sum(int(r.get('total_tokens') or 0) for r in agent_runs.values()),
    }
    return {'label': label, 'eval_id': eval_id, 'batch_id': result.batch_evaluation_id,
            'status': str(result.status), 'agg': agg, 'events': event_rows,
            'agent_perf': agent_perf, 'agent_runs': list(agent_runs.values())}


# ── OntologyPatterns KB toggle (backup → wipe → restore) ──
def _list_pattern_keys() -> list:
    """List all object keys under the OntologyPatterns S3 prefix."""
    keys = []
    for page in s3.get_paginator('list_objects_v2').paginate(Bucket=PATTERNS_BUCKET, Prefix=PATTERNS_PREFIX):
        keys.extend(o['Key'] for o in page.get('Contents', []))
    return keys


def start_patterns_ingestion() -> str:
    """Kick a re-ingestion of the OntologyPatterns KB and wait for it to finish. Returns status."""
    job = bedrock_agent.start_ingestion_job(
        knowledgeBaseId=ONTOLOGY_PATTERNS_KB_ID, dataSourceId=ONTOLOGY_PATTERNS_DS_ID)['ingestionJob']
    jid = job['ingestionJobId']
    for _ in range(40):
        time.sleep(15)
        st = bedrock_agent.get_ingestion_job(
            knowledgeBaseId=ONTOLOGY_PATTERNS_KB_ID, dataSourceId=ONTOLOGY_PATTERNS_DS_ID,
            ingestionJobId=jid)['ingestionJob']['status']
        print(f"    ingestion {jid}: {st}")
        if st in ('COMPLETE', 'FAILED', 'STOPPED'):
            return st
    return 'TIMED_OUT'


def disable_ontology_rag(*, backup_prefix: str) -> dict:
    """Back up every pattern object to backup_prefix, delete the live ones, re-ingest empty KB.

    Returns {backed_up, deleted, ingestion_status, backup_prefix}. NO-OP unless
    CONFIRM_DESTRUCTIVE_KB_WIPE is True.
    """
    if not CONFIRM_DESTRUCTIVE_KB_WIPE:
        print("  ⏭  CONFIRM_DESTRUCTIVE_KB_WIPE is False — skipping the wipe (Run B will be skipped).")
        return {'skipped': True}
    keys = _list_pattern_keys()
    print(f"  Backing up {len(keys)} pattern object(s) → s3://{PATTERNS_BUCKET}/{backup_prefix}")
    for k in keys:
        rel = k[len(PATTERNS_PREFIX):]
        s3.copy_object(Bucket=PATTERNS_BUCKET, CopySource={'Bucket': PATTERNS_BUCKET, 'Key': k},
                       Key=f"{backup_prefix}{rel}")
    # Only delete AFTER every object is safely copied.
    if keys:
        s3.delete_objects(Bucket=PATTERNS_BUCKET,
                          Delete={'Objects': [{'Key': k} for k in keys]})
    print(f"  Deleted {len(keys)} live pattern object(s); re-ingesting empty KB ...")
    st = start_patterns_ingestion()
    return {'skipped': False, 'backed_up': len(keys), 'deleted': len(keys),
            'ingestion_status': st, 'backup_prefix': backup_prefix}


def restore_ontology_rag(*, backup_prefix: str) -> dict:
    """Restore pattern objects from backup_prefix back to the live prefix, re-ingest. NO-OP if gate off."""
    if not CONFIRM_DESTRUCTIVE_KB_WIPE:
        print("  ⏭  gate off — nothing to restore.")
        return {'skipped': True}
    backup_keys = []
    for page in s3.get_paginator('list_objects_v2').paginate(Bucket=PATTERNS_BUCKET, Prefix=backup_prefix):
        backup_keys.extend(o['Key'] for o in page.get('Contents', []))
    print(f"  Restoring {len(backup_keys)} object(s) from s3://{PATTERNS_BUCKET}/{backup_prefix}")
    for k in backup_keys:
        rel = k[len(backup_prefix):]
        s3.copy_object(Bucket=PATTERNS_BUCKET, CopySource={'Bucket': PATTERNS_BUCKET, 'Key': k},
                       Key=f"{PATTERNS_PREFIX}{rel}")
    st = start_patterns_ingestion()
    return {'skipped': False, 'restored': len(backup_keys), 'ingestion_status': st}

print("✓ Helpers defined (VKG build + server-side eval + OntologyPatterns KB toggle)")
print(f"  Live OntologyPatterns objects right now: {len(_list_pattern_keys())}")

# ════════════════════════════════════════════════════════════════════════════
# k-run helpers: average a server-side arm over EVAL_K runs, and load notebook 6's
# k-run mean for the WITH-OntologyRAG arm (so it is REUSED, not re-run).
#   - Run A (WITH OntologyRAG) = the SAME ontology query agent over the SAME best-coverage
#     VKG layer that notebook 6 already evaluates k times → load its ontology_query_kmean file.
#   - Run B (WITHOUT OntologyRAG) is evaluated HERE k times over the pinned without-OntologyRAG
#     layer (WITHOUT_ONTOLOGYRAG_LAYER_ID) — no KB wipe needed (the layer is already built).
# ════════════════════════════════════════════════════════════════════════════
import glob as _glob
import statistics as _stats

EVAL_K = int(os.environ.get('EVAL_K', '3'))
_EVAL_ORDER = ['Builtin.GoalSuccessRate', 'FinalAnswerFaithfulness',
               'SqlGrounded', 'ToolCallOrdering']


def _mean(values: list):
    """Mean over non-None numeric values, rounded to 4dp (None if none present)."""
    nums = [v for v in values if isinstance(v, (int, float))]
    return round(sum(nums) / len(nums), 4) if nums else None


def run_kmean_eval(*, label: str, eval_id: str, k: int = EVAL_K) -> dict:
    """Run the server-side eval k times over `eval_id` and average the per-evaluator scores.

    Wraps run_server_side_eval(...). Returns a kmean-shaped dict (same schema as notebook 6's
    ontology_query_kmean file) so the WITHOUT arm and the reused WITH arm compare apples-to-apples.

    Parameters:
        label: human tag for this arm ('without-ontologyrag').
        eval_id: the VKG layer config id the ontology query agent resolves.
        k: number of repeat runs to average (default EVAL_K).
    Returns:
        dict with mean_scores, std_scores, agent_perf_mean, per_run_scores, batch_ids,
        per_scenario_goal_mean (by scenario_id).
    """
    runs = []
    for i in range(1, k + 1):
        print(f"\n--- {label} run {i}/{k} ---")
        runs.append(run_server_side_eval(label=label, eval_id=eval_id))

    mean_scores, std_scores = {}, {}
    for ev in _EVAL_ORDER:
        nums = [r['agg'].get(ev) for r in runs if isinstance(r['agg'].get(ev), (int, float))]
        mean_scores[ev] = round(sum(nums) / len(nums), 4) if nums else None
        std_scores[ev] = round(_stats.pstdev(nums), 4) if len(nums) > 1 else (0.0 if nums else None)

    agent_perf_mean = {
        'avg_wall_clock_s': _mean([r['agent_perf'].get('avg_wall_clock_s') for r in runs]),
        'agent_total_tokens': _mean([r['agent_perf'].get('agent_total_tokens') for r in runs]),
    }

    _goal = {}
    for r in runs:
        for e in r['events']:
            if e.get('evaluator') == 'Builtin.GoalSuccessRate' and isinstance(e.get('score'), (int, float)):
                _goal.setdefault(e.get('scenario_id'), []).append(e['score'])
    per_scenario_goal_mean = {sid: round(sum(v) / len(v), 4) for sid, v in _goal.items()}

    return {'label': label, 'eval_id': eval_id, 'eval_k': k,
            'mean_scores': mean_scores, 'std_scores': std_scores,
            'agent_perf_mean': agent_perf_mean,
            'per_run_scores': [{'batch_id': r['batch_id'], 'status': r['status'],
                                'scores': r['agg'], 'agent_perf': r['agent_perf']} for r in runs],
            'batch_ids': [r['batch_id'] for r in runs],
            'per_scenario_goal_mean': per_scenario_goal_mean}


def load_latest_kmean(prefix: str, *, label: str) -> dict:
    """Load the most recent k-run mean file results/{prefix}_*.json, tagged with `label`.

    Used to REUSE notebook 6's WITH-OntologyRAG arm (ontology_query_kmean_eval_*.json) — the
    same ontology query agent over the same best-coverage VKG layer this notebook's WITH arm
    would otherwise re-run.

    Parameters:
        prefix: filename prefix, e.g. 'ontology_query_kmean_eval'.
        label: arm tag for the comparison row ('with-ontologyrag').
    Returns:
        dict with label, mean_scores, agent_perf_mean (token key normalised), eval_id, eval_k.
    Raises:
        FileNotFoundError if no matching file exists — fail loudly so a missing notebook-6 run
        is obvious rather than silently producing a half table.
    """
    matches = sorted(_glob.glob(f"../data/eval/results/{prefix}_*.json"))
    if not matches:
        raise FileNotFoundError(
            f"No {prefix}_*.json found in ../data/eval/results/. Run notebook 6 first "
            f"(it produces the WITH-OntologyRAG arm's k-run mean).")
    path = matches[-1]
    payload = json.load(open(path))
    apm = dict(payload.get('agent_perf_mean', {}) or {})
    if 'agent_total_tokens' not in apm and 'total_tokens' in apm:
        apm['agent_total_tokens'] = apm['total_tokens']
    print(f"  \u267b [{label}] loaded {os.path.basename(path)} "
          f"(eval_k={payload.get('eval_k')}, eval_id={payload.get('eval_id')}, "
          f"scores={payload.get('mean_scores')})")
    return {'label': label,
            'mean_scores': payload.get('mean_scores', {}) or {},
            'std_scores': payload.get('std_scores', {}) or {},
            'agent_perf_mean': apm,
            'per_scenario_goal_mean': payload.get('per_scenario_goal_mean', {}) or {},
            'eval_id': payload.get('eval_id'), 'eval_k': payload.get('eval_k'),
            'source_file': os.path.basename(path)}


print(f"✓ k-run helpers ready (EVAL_K={EVAL_K}): run_kmean_eval + load_latest_kmean")


✓ Custom server-side evaluators created (LLM-as-Judge, no Lambda):
  FinalAnswerFaithfulness (SESSION) : FinalAnswerFaithfulness_abd36cca-NdYyA9DyXB
  SqlGrounded             (SESSION) : SqlGrounded_abd36cca-e3riU12Jlf
  ToolCallOrdering        (SESSION) : ToolCallOrdering_abd36cca-hkfsLBHXQ4
✓ Helpers defined (VKG build + server-side eval + OntologyPatterns KB toggle)


  Live OntologyPatterns objects right now: 471


In [8]:
# ── Layer reuse switch for the WITH-OntologyRAG arm (default: REUSE) ─────────
# Run A's "with OntologyRAG" layer is just a VKG layer built while the patterns KB is
# populated — i.e. the NORMAL/live state, exactly how notebooks 5 & 10 build their VKG
# layers. So by DEFAULT we REUSE the most recent completed VKG layer for Run A and only
# BUILD ONE NEW layer — the "without OntologyRAG" arm (Run B), which requires the
# patterns KB wiped (a state no existing layer was built in, so it must be fresh).
#
# To keep the comparison apples-to-apples, Run B builds over the SAME source tables as
# the reused Run A layer (only OntologyRAG is toggled, not the table set).
#
# Knobs (env vars):
#   REUSE_EXISTING_LAYER=0   → build a fresh WITH-OntologyRAG layer for Run A too.
#   VKG_LAYER_ID=…           → pin a specific completed VKG config `id` to reuse for Run A.
REUSE_EXISTING_LAYER = os.environ.get('REUSE_EXISTING_LAYER', '1') not in ('0', 'false', 'False', '')
VKG_LAYER_ID = os.environ.get('VKG_LAYER_ID', '').strip()
# WITHOUT-OntologyRAG arm (Run B): pin a completed VKG layer that was built WHILE the
# patterns KB was empty. When set, Run B REUSES it and SKIPS the destructive KB wipe+build
# entirely (eval only queries the finished layer — it does not need the KB wiped). This is
# the safe path: no wipe means the KB is never at risk. Unset => build fresh (needs wipe).
WITHOUT_ONTOLOGYRAG_LAYER_ID = os.environ.get('WITHOUT_ONTOLOGYRAG_LAYER_ID', '').strip()


def find_existing_vkg_layer(*, explicit_id: str = '') -> dict:
    """Locate a completed VKG layer config to reuse for the WITH-OntologyRAG arm (no rebuild).

    Parameters:
        explicit_id: optional config `id` to pin; if set it must exist, be type 'VKG',
            and have status 'completed'.
    Returns:
        A dict shaped like build_vkg_layer's output plus the layer's own dataSources and
        reused=True: {'case_id', 'status', 'duration_s', 'num_tables', 'runtime_session_id',
        'dataSources', 'reused'}.
    Raises:
        RuntimeError if no matching completed VKG layer is found — we fail loudly rather than
        silently falling back to a rebuild (which would mask a missing layer).
    """
    if explicit_id:
        item = ddb.get_item(Key={'id': explicit_id, 'version': 'v1'}).get('Item', {})
        if not item:
            raise RuntimeError(f"Pinned VKG layer id '{explicit_id}' not found in {METADATA_TABLE}")
        if item.get('type') != 'VKG':
            raise RuntimeError(f"Pinned layer '{explicit_id}' is type {item.get('type')}, expected VKG")
        if item.get('status') != 'completed':
            raise RuntimeError(
                f"Pinned VKG layer '{explicit_id}' status is {item.get('status')}, not 'completed'")
        chosen = item
    else:
        # Scan for completed VKG configs, then pick the most recently finished.
        # `type` and `status` are reserved words, so alias them in the filter expression.
        candidates: list = []
        scan_kwargs = {
            'FilterExpression': '#t = :ty AND #s = :st AND version = :v',
            'ExpressionAttributeNames': {'#t': 'type', '#s': 'status'},
            'ExpressionAttributeValues': {':ty': 'VKG', ':st': 'completed', ':v': 'v1'},
        }
        while True:
            resp = ddb.scan(**scan_kwargs)
            candidates.extend(resp.get('Items', []))
            last_key = resp.get('LastEvaluatedKey')
            if not last_key:
                break
            scan_kwargs['ExclusiveStartKey'] = last_key
        if not candidates:
            raise RuntimeError(
                f"No completed VKG layer found in {METADATA_TABLE} to reuse for Run A. "
                f"Build one (run notebook 5) or set REUSE_EXISTING_LAYER=0 to build it here."
            )
        # Prefer BEST-COVERAGE (most dataSources), then recency — a 1-table smoke-test layer
        # must not shadow the full-namespace layer the multi-table ground truth needs.
        candidates.sort(key=lambda it: (len(it.get('dataSources') or []),
                                        it.get('completedAt') or it.get('updatedAt') or ''),
                        reverse=True)
        chosen = candidates[0]
    sources = chosen.get('dataSources', []) or []
    print(f"  ♻ Reusing VKG layer {chosen['id']} for WITH-OntologyRAG arm "
          f"(name='{chosen.get('name', '?')}', {len(sources)} table(s))")
    return {'case_id': chosen['id'], 'status': 'completed', 'duration_s': 0.0,
            'num_tables': len(sources), 'runtime_session_id': str(uuid.uuid4()),
            'dataSources': sources, 'reused': True}


def prepare_with_ontologyrag_layer(*, sources: list) -> dict:
    """Reuse an existing completed VKG layer for Run A (default) or build a fresh one.

    Honours REUSE_EXISTING_LAYER + VKG_LAYER_ID. Always returns build_vkg_layer's result
    shape plus 'dataSources' (the source tables the layer covers) and a 'reused' flag, so
    Run B can build the WITHOUT arm over the SAME tables.

    Parameters:
        sources: data-source dicts (used only when building).
    Returns:
        dict with case_id, status, duration_s, num_tables, runtime_session_id,
        dataSources, reused.
    """
    if REUSE_EXISTING_LAYER:
        return find_existing_vkg_layer(explicit_id=VKG_LAYER_ID)
    out = build_vkg_layer(label='with-ontologyrag', sources=sources)
    out['reused'] = False
    out['dataSources'] = sources
    return out


_strategy = ('REUSE existing VKG layer for Run A; build ONE new layer for Run B'
             if REUSE_EXISTING_LAYER else 'BUILD fresh layers for both arms')
print(f"✓ Layer strategy: {_strategy}" + (f" (Run A pin: {VKG_LAYER_ID})" if VKG_LAYER_ID else ""))


✓ Layer strategy: REUSE existing VKG layer for Run A; build ONE new layer for Run B


## 5. Run A — VKG Build WITH OntologyRAG

The OntologyPatterns KB is populated, so the ontology agent retrieves design patterns during
generation. Build a VKG layer and evaluate the ontology query agent.

In [ ]:
print("=== RUN A: WITH OntologyRAG (reused from notebook 6's k-run mean) ===")
# Run A's WITH-OntologyRAG arm is the SAME ontology query agent over the SAME best-coverage VKG
# layer that notebook 6 already evaluates EVAL_K times (the live/populated-KB build condition).
# Re-running it here would just repeat notebook 6's work, so we REUSE its k-run mean file. Run
# notebook 6 first. (Set REUSE_NB6_WITH=0 to evaluate the WITH layer here instead.)
REUSE_NB6_WITH = os.environ.get('REUSE_NB6_WITH', '1') not in ('0', 'false', 'False', '')

if REUSE_NB6_WITH:
    kmean_a = load_latest_kmean('ontology_query_kmean_eval', label='with-ontologyrag')
    layer_a = {'case_id': kmean_a.get('eval_id'), 'status': 'completed', 'reused': True,
               'source': 'notebook-6 k-run mean', 'source_file': kmean_a.get('source_file'),
               'dataSources': SOURCES}
    print(f"  ♻ Run A reused notebook 6's VKG arm (eval_id={kmean_a.get('eval_id')}, "
          f"eval_k={kmean_a.get('eval_k')})")
else:
    layer_a = prepare_with_ontologyrag_layer(sources=SOURCES)
    if layer_a['status'] == 'completed':
        kmean_a = run_kmean_eval(label='with-ontologyrag', eval_id=layer_a['case_id'])
        kmean_a['source'] = 'evaluated in nb11'
    else:
        print("  ⚠ build did not complete — skipping Run A eval")
        kmean_a = {'label': 'with-ontologyrag', 'eval_id': layer_a.get('case_id'), 'eval_k': 0,
                   'mean_scores': {}, 'std_scores': {}, 'agent_perf_mean': {},
                   'per_run_scores': [], 'batch_ids': [], 'per_scenario_goal_mean': {}}

# Run B (if it builds fresh) must use Run A's table set so OntologyRAG on/off is the only
# variable. With the WITHOUT layer pinned (default), this is unused but kept for the build path.
RUN_B_SOURCES = layer_a.get('dataSources') or SOURCES
print(f"\n  Run A k-run MEAN scores (eval_k={kmean_a.get('eval_k')}): {kmean_a['mean_scores']}")


## 6. Disable OntologyRAG (backup → wipe → re-ingest)

Empties the OntologyPatterns KB so `retrieve_ontology_patterns` returns nothing. **Gated** by
`CONFIRM_DESTRUCTIVE_KB_WIPE` — a no-op (and Run B is skipped) unless you enabled it in the
gate cell at the top. The backup prefix is timestamped so Run-B and the restore below agree.

In [10]:
# The destructive KB wipe is only needed to BUILD a fresh without-OntologyRAG layer. If a
# completed without-OntologyRAG layer is pinned (WITHOUT_ONTOLOGYRAG_LAYER_ID), Run B reuses it
# and we SKIP the wipe entirely — the KB is never touched.
backup_prefix = f"ontology-patterns-backup/{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}/"
if WITHOUT_ONTOLOGYRAG_LAYER_ID:
    print(f"  \u267b WITHOUT_ONTOLOGYRAG_LAYER_ID pinned ({WITHOUT_ONTOLOGYRAG_LAYER_ID}) \u2014 "
          "reusing it; SKIPPING the destructive KB wipe.")
    disable_result = {'skipped': True}
else:
    disable_result = disable_ontology_rag(backup_prefix=backup_prefix)
print(f"  disable result: {disable_result}")


  Backing up 471 pattern object(s) → s3://semantic-layer-dev-artifacts-XXXXXXXXXXXX/ontology-patterns-backup/20260615-005338/


  Deleted 471 live pattern object(s); re-ingesting empty KB ...


    ingestion ZQTBABLT1D: IN_PROGRESS


    ingestion ZQTBABLT1D: IN_PROGRESS


    ingestion ZQTBABLT1D: IN_PROGRESS


...

    ingestion ZQTBABLT1D: IN_PROGRESS


    ingestion ZQTBABLT1D: IN_PROGRESS


    ingestion ZQTBABLT1D: IN_PROGRESS


    ingestion ZQTBABLT1D: IN_PROGRESS


    ingestion ZQTBABLT1D: COMPLETE
  disable result: {'skipped': False, 'backed_up': 471, 'deleted': 471, 'ingestion_status': 'COMPLETE', 'backup_prefix': 'ontology-patterns-backup/20260615-005338/'}


## 7. Run B — VKG Build WITHOUT OntologyRAG

In [ ]:
layer_b = {'status': 'skipped'}
kmean_b = {'label': 'without-ontologyrag', 'eval_id': None, 'eval_k': 0,
           'mean_scores': {}, 'std_scores': {}, 'agent_perf_mean': {},
           'per_run_scores': [], 'batch_ids': [], 'per_scenario_goal_mean': {}}
restore_result = {'skipped': True}

if WITHOUT_ONTOLOGYRAG_LAYER_ID:
    # SAFE PATH (default): reuse a completed without-OntologyRAG layer and evaluate it k times.
    # No KB wipe happened (the disable cell skipped it), so there is nothing to restore.
    print("=== RUN B: WITHOUT OntologyRAG (k-run over pinned layer) ===")
    _item = ddb.get_item(Key={'id': WITHOUT_ONTOLOGYRAG_LAYER_ID, 'version': 'v1'}).get('Item', {})
    if not _item:
        raise RuntimeError(f"Pinned WITHOUT layer '{WITHOUT_ONTOLOGYRAG_LAYER_ID}' not found in {METADATA_TABLE}")
    if _item.get('type') != 'VKG':
        raise RuntimeError(f"Pinned WITHOUT layer '{WITHOUT_ONTOLOGYRAG_LAYER_ID}' is type {_item.get('type')}, expected VKG")
    if _item.get('status') != 'completed':
        raise RuntimeError(f"Pinned WITHOUT layer '{WITHOUT_ONTOLOGYRAG_LAYER_ID}' status is {_item.get('status')}, not 'completed'")
    layer_b = {'case_id': WITHOUT_ONTOLOGYRAG_LAYER_ID, 'status': 'completed',
               'num_tables': len(_item.get('dataSources', []) or []), 'reused': True}
    print(f"  ♻ Reusing without-OntologyRAG VKG layer {WITHOUT_ONTOLOGYRAG_LAYER_ID} "
          f"(name='{_item.get('name','?')}', {layer_b['num_tables']} table(s)) — evaluating k times")
    kmean_b = run_kmean_eval(label='without-ontologyrag', eval_id=layer_b['case_id'])
elif CONFIRM_DESTRUCTIVE_KB_WIPE and not disable_result.get('skipped'):
    # FRESH-BUILD PATH: the OntologyPatterns KB is WIPED at this point (disable cell). Only the
    # BUILD agent reads that KB (via retrieve_ontology_patterns) — the query eval does NOT. So we
    # build inside a try/finally that RESTORES the KB the instant the build finishes, THEN run the
    # (KB-independent) k-run eval with the patterns already back. This keeps the destructive window
    # to just the build, not the build + k× eval — much safer (and survives an eval-phase crash).
    try:
        print("=== RUN B: WITHOUT OntologyRAG (building fresh) ===")
        layer_b = build_vkg_layer(label='without-ontologyrag', sources=RUN_B_SOURCES)
    finally:
        # ALWAYS restore right after the build — before the long eval, and even if the build raised.
        restore_result = restore_ontology_rag(backup_prefix=backup_prefix)
        print(f"  restore result: {restore_result}")
        print(f"  Live OntologyPatterns objects after restore: {len(_list_pattern_keys())}")
    # Eval the freshly-built layer k times (KB is restored now — the eval never touches it).
    if layer_b.get('status') == 'completed':
        print(f"  ✓ Built without-OntologyRAG layer {layer_b['case_id']} — now evaluating k times.")
        print(f"    (Pin WITHOUT_ONTOLOGYRAG_LAYER_ID={layer_b['case_id']} in .env to reuse it next time.)")
        kmean_b = run_kmean_eval(label='without-ontologyrag', eval_id=layer_b['case_id'])
    else:
        print("  ⚠ build did not complete — skipping Run B eval.")
else:
    print("⏭  Run B skipped. Pin WITHOUT_ONTOLOGYRAG_LAYER_ID to reuse a completed layer (no "
          "wipe — recommended), or enable CONFIRM_DESTRUCTIVE_KB_WIPE to build one fresh.")

print(f"\n  Run B k-run MEAN scores (eval_k={kmean_b.get('eval_k')}): {kmean_b['mean_scores']}")


## 8. Restore OntologyRAG

Always run this after Run B to put the patterns back and re-ingest. Safe/no-op if the gate is
off or nothing was wiped.

In [12]:
# Restore now runs inside the Run B cell's finally block (so it executes even if Run B raised).
# This cell only RE-VERIFIES the KB is whole — it does not restore again (idempotent check).
if CONFIRM_DESTRUCTIVE_KB_WIPE and not disable_result.get('skipped'):
    _live = len(_list_pattern_keys())
    print(f"  \u2713 OntologyPatterns restore verified: {_live} live object(s) (expected 471).")
    if _live == 0:
        print(f"  \u26a0 KB is EMPTY \u2014 restore did not run; restore manually from {backup_prefix}")
else:
    print("  \u23ed  gate off \u2014 nothing was wiped, nothing to restore.")


  Restoring 471 object(s) from s3://semantic-layer-dev-artifacts-XXXXXXXXXXXX/ontology-patterns-backup/20260615-005338/


    ingestion 0WMVI2JXIV: IN_PROGRESS


    ingestion 0WMVI2JXIV: IN_PROGRESS


    ingestion 0WMVI2JXIV: IN_PROGRESS


    ingestion 0WMVI2JXIV: IN_PROGRESS


...

    ingestion 0WMVI2JXIV: IN_PROGRESS


    ingestion 0WMVI2JXIV: IN_PROGRESS


    ingestion 0WMVI2JXIV: IN_PROGRESS


    ingestion 0WMVI2JXIV: COMPLETE
  restore result: {'skipped': False, 'restored': 471, 'ingestion_status': 'COMPLETE'}


  Live OntologyPatterns objects after restore: 471


## 9. Compare the Two Outcomes

In [ ]:
_EVALS = ['Builtin.GoalSuccessRate',
          'FinalAnswerFaithfulness', 'SqlGrounded', 'ToolCallOrdering']


def summarize_kmean(kmean: dict) -> dict:
    """Flatten one arm's k-run MEAN result into a comparison row.

    `kmean` is the dict from run_kmean_eval / load_latest_kmean — per-evaluator mean scores in
    `mean_scores`, plus the agent's own mean latency/tokens in `agent_perf_mean`.
    """
    scores = kmean.get('mean_scores', {}) or {}
    perf = kmean.get('agent_perf_mean', {}) or {}
    row = {'arm': kmean.get('label'), 'eval_k': kmean.get('eval_k')}
    row.update({ev: scores.get(ev) for ev in _EVALS})
    row['avg_latency_s'] = perf.get('avg_wall_clock_s')
    row['agent_total_tokens'] = perf.get('agent_total_tokens')
    return row


comparison = pd.DataFrame([summarize_kmean(kmean_a), summarize_kmean(kmean_b)])
print("=== COMPARISON (k-run MEAN): VKG with vs without OntologyRAG ===")
display(comparison)

os.makedirs('../data/eval/results', exist_ok=True)
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
out = f'../data/eval/results/ontology_rag_on_off_kmean_{ts}.json'
with open(out, 'w') as f:
    json.dump(
        _redact_account_ids({
            'eval_k': {'with': kmean_a.get('eval_k'), 'without': kmean_b.get('eval_k')},
            'with_source': kmean_a.get('source', kmean_a.get('source_file')),
            'eval_ids': {'with': kmean_a.get('eval_id'), 'without': kmean_b.get('eval_id')},
            'layer_a': layer_a, 'layer_b': layer_b,
            'disable_result': disable_result, 'restore_result': restore_result,
            'kmean_a': kmean_a, 'kmean_b': kmean_b,
            'comparison': comparison.to_dict(orient='records'),
            'custom_evaluators': {'FinalAnswerFaithfulness': ANSWER_FAITHFUL_ID,
                                  'SqlGrounded': SQL_GROUNDED_ID, 'ToolCallOrdering': TOOL_ORDERING_ID},
        }),
        f, indent=2, default=str,
    )
print(f"✓ Wrote {out}")

# Headline — GoalSuccessRate (k-run mean) is the primary accuracy signal. The OntologyRAG effect
# is smaller than the cross-run noise; multiple runs per arm (EVAL_K) average that out. VKG's
# SqlGrounded / FinalAnswerFaithfulness sit in the documented harvest-noise band.
a_s, b_s = kmean_a.get('mean_scores', {}) or {}, kmean_b.get('mean_scores', {}) or {}
a_gsr, b_gsr = a_s.get('Builtin.GoalSuccessRate'), b_s.get('Builtin.GoalSuccessRate')
if a_gsr is not None and b_gsr is not None:
    winner = 'with-ontologyrag' if a_gsr > b_gsr else ('without-ontologyrag' if b_gsr > a_gsr else 'tie')
    a_perf, b_perf = kmean_a.get('agent_perf_mean', {}), kmean_b.get('agent_perf_mean', {})
    headline = (
        "### Headline (k-run mean)\n"
        f"- **Runs averaged:** with {kmean_a.get('eval_k')} (reused from nb6) · without {kmean_b.get('eval_k')}\n"
        f"- **GoalSuccessRate:** with {a_gsr:.1%} vs without {b_gsr:.1%} → **{winner}**\n"
        f"- **FinalAnswerFaithfulness:** with {a_s.get('FinalAnswerFaithfulness')} vs without {b_s.get('FinalAnswerFaithfulness')}\n"
        f"- **SqlGrounded:** with {a_s.get('SqlGrounded')} vs without {b_s.get('SqlGrounded')}\n"
        f"- **ToolCallOrdering:** with {a_s.get('ToolCallOrdering')} vs without {b_s.get('ToolCallOrdering')}\n"
        f"- **Avg latency:** with {a_perf.get('avg_wall_clock_s')}s vs without {b_perf.get('avg_wall_clock_s')}s\n"
        f"- **Agent tokens:** with {a_perf.get('agent_total_tokens')} vs without {b_perf.get('agent_total_tokens')}\n"
        f"\n_The OntologyRAG effect is near the cross-run noise floor; even averaged over "
        f"{kmean_b.get('eval_k')} run(s), read this as directional, not decisive. Judge VKG by "
        f"GoalSuccessRate + answer inspection (SqlGrounded/FAF sit in the harvest-noise band)._"
    )
    display(Markdown(headline))
else:
    print("⚠ Comparison incomplete — Run B was skipped (no pinned WITHOUT layer + gate off) "
          "or a build failed.")


## Summary

Compares a VKG semantic layer built **with** OntologyRAG (ontology design patterns retrieved
from a Bedrock KB) against one built **without** (patterns KB emptied), both evaluated via the
ontology query agent over the groundtruth dataset.

### Layer reuse (default)
The WITH-OntologyRAG layer is an ordinary VKG layer (built while the patterns KB is populated —
the normal/live condition), so Run A **reuses** the most recent completed VKG layer by default.
Only the WITHOUT-OntologyRAG layer is genuinely new — it must be built with the patterns KB
wiped — so a run builds **exactly one** layer. Run B builds over Run A's table set, so the only
variable between arms is OntologyRAG on/off.

### Safety design (destructive step)
Run B empties the OntologyPatterns KB S3 data source. This notebook makes that reversible:
**backup → delete → re-ingest empty → run → restore from backup → re-ingest**. The destructive
cells are gated behind `CONFIRM_DESTRUCTIVE_KB_WIPE` (default `False`), so by default only
Run A executes and the comparison reports Run B as skipped. Enable the flag only when you
intend to run the without-OntologyRAG arm, and verify the backup count before the delete.

### Knobs
- `REUSE_EXISTING_LAYER` (default `1`) — reuse the most recent completed VKG layer for Run A
  instead of rebuilding. Set `0` to build a fresh WITH-OntologyRAG layer in this notebook.
- `VKG_LAYER_ID` — pin a specific completed VKG config `id` to reuse for Run A (overrides the
  "most recent" auto-pick).
- `MAX_TABLES` (default 0) — tables discovered for a **fresh** build (`REUSE_EXISTING_LAYER=0`);
  when reusing, Run A's table set comes from the reused layer and Run B follows it. 0 = all.
- `MAX_ROWS` (default 0); 0 = all.

### Notes
- Always run the restore cell (§8) after Run B; it's a safe no-op when the gate is off.
